## Milestone 4 — PVDN (`Milestone4` branch)

1. **Reference (below):** Milestone 3 preprocessing + Model 1 — run cells in order on Expanse.
2. **Tasks 1–6 (after Reference):** Placeholders — add PCA / Model 2, evaluation, README notes, conclusion, test predictions.

`milestone_3.ipynb` is removed on this branch; the Reference block is the M3 copy.


---
## Reference — Milestone 3 pipeline & Model 1

Same pipeline as Milestone 3 (through RF1 + evaluation). Outputs under `_m4_outputs`.


## 1. Complete Preprocessing using Spark

### Libraries & Setup

In [ ]:
import os
import json
import glob as pyglob
import pandas as pd
from functools import reduce
from IPython.display import display, Markdown

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    BooleanType, ArrayType, DoubleType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def show(df, n=20):
    display(df.limit(n).toPandas())


In [ ]:
# Expanse Environment (active)
DATA_ROOT = "/expanse/lustre/projects/uci157/kkravchenko/provident-vehicle-detection-at-night-pvdn"

# Local Environment
# import kagglehub
# DATA_ROOT = kagglehub.dataset_download("saralajew/provident-vehicle-detection-at-night-pvdn")
# os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
# os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

OUTPUT_ROOT = os.path.join(DATA_ROOT, "_m4_outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)


In [ ]:
# Expanse Environment (active)
spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

# Local Environment
# spark = SparkSession.builder \
#     .appName("PVDN Milestone 4") \
#     .master("local[*]") \
#     .config("spark.driver.memory", "12g") \
#     .config("spark.executor.memory", "12g") \
#     .config("spark.sql.shuffle.partitions", 10) \
#     .config("spark.default.parallelism", 10) \
#     .config("spark.driver.maxResultSize", "4g") \
#     .config("spark.sql.adaptive.enabled", "true") \
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
#     .getOrCreate()

spark


### Load Raw Data

In [ ]:
base = os.path.join(DATA_ROOT, "PVDN")
ann_paths = sorted(pyglob.glob(os.path.join(base, "*", "*", "labels", "image_annotations.json")))
seq_paths = sorted(pyglob.glob(os.path.join(base, "*", "*", "labels", "sequences.json")))

def add_path_metadata(df, tod_offset, split_offset):
    return df \
        .withColumn("_parts", F.split(F.input_file_name(), "/")) \
        .withColumn("time_of_day", F.element_at(F.col("_parts"), tod_offset)) \
        .withColumn("split", F.element_at(F.col("_parts"), split_offset)) \
        .drop("_parts")

ann_raw = spark.read.option("multiLine", True).json(ann_paths)

images_raw = add_path_metadata(
    ann_raw.select(F.explode("images").alias("img")).select(
        F.col("img.id").alias("image_id"),
        F.col("img.file_name").alias("file_name"),
        F.col("img.height").alias("height"),
        F.col("img.width").alias("width"),
        F.col("img.timestamp").alias("timestamp"),
        F.col("img.camera_configuration").alias("camera_configuration"),
    ), tod_offset=-4, split_offset=-3
)

annotations_df = add_path_metadata(
    ann_raw.select(F.explode("annotations").alias("ann")).select(
        F.col("ann.id").alias("annotation_id"),
        F.col("ann.image_id").alias("image_id"),
        F.col("ann.category").alias("category"),
    ), tod_offset=-4, split_offset=-3
)

images_df = images_raw.join(annotations_df.select("image_id", "category"), on="image_id", how="left")
print(f"images_df rows: {images_df.count()}")
images_df.printSchema()


In [ ]:
seq_raw = spark.read.option("multiLine", True).json(seq_paths)

sequences_df = add_path_metadata(
    seq_raw.select(F.explode("sequences").alias("seq")).select(
        F.col("seq.id").alias("sequence_id"),
        F.col("seq.image_ids").alias("image_ids"),
        F.col("seq.weather").alias("weather"),
        F.col("seq.road_type").alias("road_type"),
        F.col("seq.direction").alias("direction"),
        F.col("seq.view").alias("view"),
        F.col("seq.street_style").alias("street_style"),
        F.col("seq.proband_behaviour").alias("proband_behaviour"),
        F.col("seq.environment_lighting").alias("environment_lighting"),
        F.col("seq.dome").alias("dome"),
    ), tod_offset=-4, split_offset=-3
)

# Explode image_ids to get one row per image in each sequence
seq_image_map = sequences_df.select(
    F.explode("image_ids").alias("image_id"),
    "weather", "road_type", "direction", "view",
    "street_style", "proband_behaviour", "environment_lighting", "dome",
)

print(f"sequences_df rows: {sequences_df.count()}")
sequences_df.printSchema()


In [ ]:
kp_dirs = sorted(pyglob.glob(os.path.join(base, "*", "*", "labels", "keypoints")))

vehicle_rows = []
instance_rows = []
blur_rows = []

for kp_dir in kp_dirs:
    parts = kp_dir.replace("\\", "/").split("/")
    time_of_day = parts[-4]
    split_name = parts[-3]
    for fp in sorted(pyglob.glob(os.path.join(kp_dir, "*.json"))):
        with open(fp) as f:
            rec = json.load(f)
        img_id = rec["image_id"]
        for blur in rec.get("blurs", []):
            blur_rows.append((img_id, float(blur[0]), float(blur[1]),
                              float(blur[2]), float(blur[3])))
        for veh in rec.get("annotations", []):
            insts = veh.get("instances", [])
            vehicle_rows.append((
                img_id, veh["pos"], veh["oid"], bool(veh["direct"]), len(insts)
            ))
            for inst in insts:
                instance_rows.append((
                    img_id, veh["oid"], bool(veh["direct"]),
                    inst["pos"], inst["iid"],
                    bool(inst["direct"]), bool(inst["rear"])
                ))

vehicles_df = spark.createDataFrame(vehicle_rows, schema=StructType([
    StructField("image_id", LongType()),
    StructField("vehicle_pos", ArrayType(IntegerType())),
    StructField("vehicle_oid", LongType()),
    StructField("vehicle_direct", BooleanType()),
    StructField("num_instances", IntegerType()),
]))

instances_df = spark.createDataFrame(instance_rows, schema=StructType([
    StructField("image_id", LongType()),
    StructField("vehicle_oid", LongType()),
    StructField("vehicle_direct", BooleanType()),
    StructField("instance_pos", ArrayType(IntegerType())),
    StructField("instance_id", LongType()),
    StructField("instance_direct", BooleanType()),
    StructField("instance_rear", BooleanType()),
]))

blurs_df = spark.createDataFrame(blur_rows, schema=StructType([
    StructField("image_id", LongType()),
    StructField("x1", DoubleType()),
    StructField("y1", DoubleType()),
    StructField("x2", DoubleType()),
    StructField("y2", DoubleType()),
]))

print(f"vehicles_df rows:  {vehicles_df.count()}")
print(f"instances_df rows: {instances_df.count()}")
print(f"blurs_df rows:     {blurs_df.count()}")


### Feature Engineering

| Feature Group | Source | Features |
|---|---|---|
| Vehicle aggregations | `vehicles_df` | `num_vehicles`, `mean_veh_x`, `mean_veh_y`, `num_direct_vehicles` |
| Light instance aggregations | `instances_df` | `num_light_instances`, `mean_inst_x`, `mean_inst_y`, `num_direct_lights`, `num_rear_lights` |
| Blur aggregations | `blurs_df` | `num_blur_regions`, `mean_blur_area` |
| Sequence metadata | `seq_image_map` | `weather`, `road_type`, `direction`, `view`, `street_style`, `proband_behaviour`, `environment_lighting`, `dome` |

In [ ]:
veh_agg = vehicles_df.select(
    "image_id",
    F.col("vehicle_pos").getItem(0).alias("veh_x"),
    F.col("vehicle_pos").getItem(1).alias("veh_y"),
    "vehicle_direct",
).groupBy("image_id").agg(
    F.count("*").alias("num_vehicles"),
    F.mean("veh_x").alias("mean_veh_x"),
    F.mean("veh_y").alias("mean_veh_y"),
    F.sum(F.when(F.col("vehicle_direct"), 1).otherwise(0)).alias("num_direct_vehicles"),
)

inst_agg = instances_df.select(
    "image_id",
    F.col("instance_pos").getItem(0).alias("inst_x"),
    F.col("instance_pos").getItem(1).alias("inst_y"),
    "instance_direct",
    "instance_rear",
).groupBy("image_id").agg(
    F.count("*").alias("num_light_instances"),
    F.mean("inst_x").alias("mean_inst_x"),
    F.mean("inst_y").alias("mean_inst_y"),
    F.sum(F.when(F.col("instance_direct"), 1).otherwise(0)).alias("num_direct_lights"),
    F.sum(F.when(F.col("instance_rear"), 1).otherwise(0)).alias("num_rear_lights"),
)

blur_agg = blurs_df.withColumn(
    "blur_area", (F.col("x2") - F.col("x1")) * (F.col("y2") - F.col("y1"))
).groupBy("image_id").agg(
    F.count("*").alias("num_blur_regions"),
    F.mean("blur_area").alias("mean_blur_area"),
)

print("Vehicle aggregations:")
show(veh_agg, 5)
print("Instance aggregations:")
show(inst_agg, 5)
print("Blur aggregations:")
show(blur_agg, 5)


In [ ]:
model_df = images_df.select("image_id", "category", "split") \
    .join(veh_agg,  on="image_id", how="left") \
    .join(inst_agg, on="image_id", how="left") \
    .join(blur_agg, on="image_id", how="left") \
    .join(seq_image_map, on="image_id", how="left")

print(f"model_df rows: {model_df.count()}")
model_df.printSchema()


### Preprocessing Pipeline

| Stage | Transformer | Purpose |
|---|---|---|
| 1 | `Imputer` (mean strategy) | Fill NaN in numeric columns for images with no vehicles/blurs |
| 2 | `StringIndexer` × 8 | Encode integer-coded categorical fields to label indices |
| 3 | `OneHotEncoder` | Expand label indices to sparse binary vectors |
| 4 | `VectorAssembler` | Concatenate all numeric + encoded features into one feature vector |
| 5 | `StandardScaler` | Normalize to zero mean, unit variance |

In [ ]:
categorical_cols = [
    "weather", "road_type", "direction", "view",
    "street_style", "proband_behaviour", "environment_lighting", "dome"
]
for c in categorical_cols:
    model_df = model_df.withColumn(c, F.col(c).cast(StringType()))

numeric_cols = [
    # num_vehicles and num_direct_vehicles are excluded to prevent data leakage:
    # label_reflection = (num_vehicles - num_direct_vehicles) > 0
    "mean_veh_x", "mean_veh_y",
    "num_light_instances", "mean_inst_x", "mean_inst_y", "num_direct_lights", "num_rear_lights",
    "num_blur_regions", "mean_blur_area",
]

for c in numeric_cols:
    model_df = model_df.withColumn(c, F.col(c).cast(DoubleType()))

print("Sample rows from model_df:")
show(model_df.select(["image_id", "category"] + numeric_cols[:5]), 5)


In [ ]:
imputed_cols = [c + "_imputed" for c in numeric_cols]
imputer = Imputer(
    strategy="mean",
    inputCols=numeric_cols,
    outputCols=imputed_cols,
)

indexed_cols = [c + "_idx" for c in categorical_cols]
string_indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep",
    )
    for c in categorical_cols
]

ohe_cols = [c + "_ohe" for c in categorical_cols]
ohe = OneHotEncoder(
    inputCols=indexed_cols,
    outputCols=ohe_cols,
    dropLast=True,
)

assembler = VectorAssembler(
    inputCols=imputed_cols + ohe_cols,
    outputCol="features_raw",
    handleInvalid="keep",
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)

preprocessing_pipeline = Pipeline(stages=[
    imputer,
    *string_indexers,
    ohe,
    assembler,
    scaler,
])

print(f"Pipeline stages: {len(preprocessing_pipeline.getStages())}")


### Train / Test Split

In [ ]:

train_df = model_df.filter(F.col("split") == "train")
test_df  = model_df.filter(F.col("split") == "test")

print(f"Train rows: {train_df.count()}")
print(f"Test rows:  {test_df.count()}")

display(Markdown("**Train category distribution:**"))
show(train_df.groupBy("category").agg(F.count("*").alias("count")).orderBy("category"))


### Class Weights

In [ ]:
n_train = train_df.count()
n_classes = train_df.select("category").distinct().count()

class_counts = train_df.groupBy("category").agg(F.count("*").alias("count"))
class_weights = class_counts.withColumn(
    "weight",
    F.lit(n_train) / (F.lit(n_classes) * F.col("count"))
)

display(Markdown("**Class weights (inverse-frequency):**"))
show(class_weights.orderBy("category"))

# Join weights onto train/test sets
train_df = train_df.join(class_weights.select("category", "weight"), on="category", how="left")
test_df  = test_df.join(class_weights.select("category", "weight"), on="category", how="left")


### Fit Preprocessing Pipeline & Transform Data

In [ ]:
preprocessing_model = preprocessing_pipeline.fit(train_df)

train_transformed = preprocessing_model.transform(train_df)
test_transformed  = preprocessing_model.transform(test_df)

print("Transformed training sample (features vector):")
show(train_transformed.select("image_id", "category", "features"), 5)


### Verify Preprocessing

In [ ]:
null_checks = train_transformed.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in imputed_cols
])

display(Markdown("**Null counts after Imputer (should all be 0):**"))
show(null_checks)

print(f"Feature vector dimensionality: {len(train_transformed.select('features').first()[0])}")


## 2. Train Your First Distributed Model

### Model Training

In [ ]:
val_df = model_df.filter(F.col("split") == "val")

val_transformed = preprocessing_model.transform(val_df)
#    reflection present = at least one indirect (not-direct) vehicle
#    indirect = num_vehicles - num_direct_vehicles

def add_reflection_label(df):
    num_vehicles = F.coalesce(F.col("num_vehicles"), F.lit(0.0))
    num_direct   = F.coalesce(F.col("num_direct_vehicles"), F.lit(0.0))
    num_indirect = num_vehicles - num_direct

    return (df
        .withColumn("num_indirect_vehicles", num_indirect)
        .withColumn("label_reflection", (F.col("num_indirect_vehicles") > 0).cast("double"))
    )

train_ref = add_reflection_label(train_transformed)
val_ref   = add_reflection_label(val_transformed)
test_ref  = add_reflection_label(test_transformed)

print("Reflection label distribution (TRAIN):")
train_ref.groupBy("label_reflection").count().show()

#  Class weights for imbalance (informative + robust)
n_train = train_ref.count()
label_counts = train_ref.groupBy("label_reflection").count()

# weight = N / (K * count(label))
K = 2
label_weights = label_counts.withColumn(
    "sample_weight",
    F.lit(n_train) / (F.lit(K) * F.col("count"))
).select("label_reflection", "sample_weight")

train_ref = train_ref.join(label_weights, on="label_reflection", how="left")
val_ref   = val_ref.join(label_weights, on="label_reflection", how="left")
test_ref  = test_ref.join(label_weights, on="label_reflection", how="left")

rf1 = RandomForestClassifier(
    labelCol="label_reflection",
    featuresCol="features",
    weightCol="sample_weight",
    numTrees=200,
    maxDepth=8,
    seed=42
)

rf2 = RandomForestClassifier(
    labelCol="label_reflection",
    featuresCol="features",
    weightCol="sample_weight",
    numTrees=500,
    maxDepth=16,
    seed=42
)

model_rf1 = rf1.fit(train_ref)
model_rf2 = rf2.fit(train_ref)

print("Trained RF1 and RF2 for light reflection prediction.")


### Evaluation

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol="label_reflection",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label_reflection",
    predictionCol="prediction",
    metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label_reflection",
    predictionCol="prediction",
    metricName="f1"
)

eval_results = []

def evaluate(model, df, model_label, split_name):
    pred = model.transform(df)
    auc = auc_eval.evaluate(pred)
    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    eval_results.append({"Model": model_label, "Split": split_name,
                         "AUC": round(auc, 4), "Accuracy": round(acc, 4), "F1": round(f1, 4)})
    print(f"{split_name}: AUC={auc:.4f} | Acc={acc:.4f} | F1={f1:.4f}")
    return pred

print("=== RF1 (numTrees=200, maxDepth=8) ===")
pred_train_1 = evaluate(model_rf1, train_ref, "RF1 (numTrees=200, maxDepth=8)",  "Train")
pred_val_1   = evaluate(model_rf1, val_ref,   "RF1 (numTrees=200, maxDepth=8)",  "Val")
pred_test_1  = evaluate(model_rf1, test_ref,  "RF1 (numTrees=200, maxDepth=8)",  "Test")

print("\n=== RF2 (numTrees=500, maxDepth=16) ===")
pred_train_2 = evaluate(model_rf2, train_ref, "RF2 (numTrees=500, maxDepth=16)", "Train")
pred_val_2   = evaluate(model_rf2, val_ref,   "RF2 (numTrees=500, maxDepth=16)", "Val")
pred_test_2  = evaluate(model_rf2, test_ref,  "RF2 (numTrees=500, maxDepth=16)", "Test")


### Training vs. Test Error

In [ ]:
metrics_df = pd.DataFrame(eval_results)
display(Markdown("**Model Metrics — Train / Val / Test:**"))
display(metrics_df)



In [ ]:
df_err = pd.DataFrame(eval_results)
df_train = df_err[df_err["Split"] == "Train"][["Model", "Accuracy", "F1"]].rename(
    columns={"Accuracy": "Train Acc", "F1": "Train F1"})
df_test  = df_err[df_err["Split"] == "Test"][["Model", "Accuracy", "F1"]].rename(
    columns={"Accuracy": "Test Acc", "F1": "Test F1"})

error_df = df_train.merge(df_test, on="Model")
error_df["Train Error"] = (1 - error_df["Train Acc"]).round(4)
error_df["Test Error"]  = (1 - error_df["Test Acc"]).round(4)
error_df["Error Gap (Test−Train)"] = (error_df["Test Error"] - error_df["Train Error"]).round(4)
error_df = error_df[["Model", "Train Error", "Test Error", "Error Gap (Test−Train)", "Train F1", "Test F1"]]

display(Markdown("**Training vs. Test Error Comparison:**"))
display(error_df)



### Ground Truth vs. Predictions

In [ ]:
from pyspark.ml.functions import vector_to_array

display(Markdown("**Train Set — Ground Truth vs. Predictions (RF1, sample of 25):**"))
train_sample = pred_train_1.select(
    "image_id",
    F.col("label_reflection").alias("ground_truth"),
    F.col("prediction"),
    F.round(vector_to_array(F.col("probability")).getItem(1), 4).alias("prob_reflection"),
).limit(25).toPandas()
display(train_sample)

correct   = (train_sample["ground_truth"] == train_sample["prediction"]).sum()
incorrect = len(train_sample) - correct
print(f"Correct: {correct}/25  |  Incorrect: {incorrect}/25")



In [ ]:
display(Markdown("**Test Set — Ground Truth vs. Predictions (RF1, sample of 25):**"))
test_sample = pred_test_1.select(
    "image_id",
    F.col("label_reflection").alias("ground_truth"),
    F.col("prediction"),
    F.round(vector_to_array(F.col("probability")).getItem(1), 4).alias("prob_reflection"),
).limit(25).toPandas()
display(test_sample)

correct   = (test_sample["ground_truth"] == test_sample["prediction"]).sum()
incorrect = len(test_sample) - correct
print(f"Correct: {correct}/25  |  Incorrect: {incorrect}/25")



## 3. Fitting Analysis

### 3a. Where Does the Model Fit? 
To determine whether the model is underfitting or overfitting, we compare train, validation, and test performance across two RandomForest configurations.

### Model 1 — RF1 (numTrees = 200, maxDepth = 8)

- Train: AUC = 0.9827 | Accuracy = 0.9225 | F1 = 0.9232  
- Validation: AUC = 0.9806 | Accuracy = 0.9224 | F1 = 0.9238  
- Test: AUC = 0.9612 | Accuracy = 0.9116 | F1 = 0.9122  

Train–Test Accuracy Gap = **0.0109**

This very small performance gap indicates that RF1 generalizes well and shows **low variance**, meaning it is **well-fit** rather than underfitting or overfitting.

### Model 2 — RF2 (numTrees = 500, maxDepth = 16)

- Train: AUC = 0.9994 | Accuracy = 0.9890 | F1 = 0.9890  
- Validation: AUC = 0.9799 | Accuracy = 0.9344 | F1 = 0.9344  
- Test: AUC = 0.9609 | Accuracy = 0.8969 | F1 = 0.8961  

Train–Test Accuracy Gap = **0.0921**

Although RF2 achieves nearly perfect training performance, the larger gap between train and test metrics indicates **significant overfitting**. The deeper trees and larger number of trees increase model complexity, allowing the model to memorize training patterns that do not generalize as well to unseen data.

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_fit = pd.DataFrame(eval_results)
splits = ["Train", "Val", "Test"]
models = list(df_fit["Model"].unique())
x = np.arange(len(splits))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ["Accuracy", "F1"]):
    for i, model in enumerate(models):
        vals = [df_fit[(df_fit["Model"] == model) & (df_fit["Split"] == s)][metric].values[0]
                for s in splits]
        ax.bar(x + i * width, vals, width, label=model)
    ax.set_xlabel("Split")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} by Split and Model")
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(splits)
    ax.set_ylim(0.80, 1.01)
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Fitting Analysis: Train / Val / Test Performance", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

display(Markdown("**Fitting Summary:**"))
for model in models:
    train_acc = df_fit[(df_fit["Model"] == model) & (df_fit["Split"] == "Train")]["Accuracy"].values[0]
    val_acc   = df_fit[(df_fit["Model"] == model) & (df_fit["Split"] == "Val")]["Accuracy"].values[0]
    test_acc  = df_fit[(df_fit["Model"] == model) & (df_fit["Split"] == "Test")]["Accuracy"].values[0]
    gap = round(train_acc - test_acc, 4)
    if gap < 0.02:
        diagnosis = "Well-fitting (low variance, low bias)"
    elif gap < 0.06:
        diagnosis = "Moderate overfitting (some variance)"
    else:
        diagnosis = "Significant overfitting (high variance)"
    print(f"{model}")
    print(f"  Train={train_acc:.4f}  Val={val_acc:.4f}  Test={test_acc:.4f}  Train-Test Gap={gap:.4f}")
    print(f"  → {diagnosis}\n")



### 3b. Alternative Model with Different Hyperparameters

In [ ]:
df_compare = pd.DataFrame(eval_results)

display(Markdown("**Baseline (RF1) vs. Alternative (RF2) — Full Comparison:**"))
pivot = df_compare.pivot(index="Split", columns="Model", values=["AUC", "Accuracy", "F1"])
pivot = pivot.reindex(["Train", "Val", "Test"])
display(pivot)

display(Markdown("**Error Gap (Train − Test Accuracy):**"))
for model in df_compare["Model"].unique():
    train = df_compare[(df_compare["Model"] == model) & (df_compare["Split"] == "Train")]["Accuracy"].values[0]
    test  = df_compare[(df_compare["Model"] == model) & (df_compare["Split"] == "Test")]["Accuracy"].values[0]
    print(f"  {model}: gap = {round(train - test, 4)}")



### 3c. Which Model Performs Best and Why?
Although RF2 has higher training accuracy, RF1 performs better overall because it maintains much more stable validation and test performance.

RF1 achieves:
- Better generalization
- Lower variance
- More consistent F1 across all splits

RF2 demonstrates stronger fitting capacity but suffers from reduced test performance, indicating that increased complexity did not improve generalization.

Therefore, **RF1 is selected as the better model for Milestone 3**.

### 3d. Next Models for Milestone 4
For Milestone 4, our group plans on using:

1. **Gradient Boosted Trees (Spark MLlib GBTClassifier)**  
   - Can improve predictive performance by iteratively correcting previous tree errors.
   - Often performs better than RandomForest on structured tabular data.

2. **Distributed XGBoost (SparkXGBClassifier or Ray Train XGBoostTrainer)**  
   - Provides stronger regularization and more advanced boosting strategies.
   - Commonly achieves high performance on classification tasks with engineered features.


## 4. Conclusion Section

### What is the conclusion of your 1st model?
The first distributed model implemented for this project was a Spark MLlib RandomForestClassifier trained on SDSC Expanse to predict reflection-related vehicle visibility patterns.

The baseline model (RF1) achieved strong and stable performance:
- Test Accuracy = 0.9116
- Test F1 = 0.9122
- Test AUC = 0.9612

These results indicate that the engineered Spark features provide strong predictive information and that the model generalizes well to unseen data.

A more complex model (RF2) achieved higher training accuracy but showed clear overfitting, demonstrating that increased model complexity does not necessarily improve generalization.

### What can be done to possibly improve it?
To improve future performance, several steps can be taken:

- Additional feature engineering using richer lighting and instance-level statistics
- Further hyperparameter tuning (maxDepth, numTrees, minInstancesPerNode, featureSubsetStrategy)
- Improved class imbalance handling using weighted sampling or threshold tuning
- Testing stronger boosting-based models such as Gradient Boosted Trees or XGBoost

### How did distributed computing help with this task?
Distributed computing on SDSC Expanse was essential for this milestone because Spark allowed:

- Parallel preprocessing across tens of thousands of images
- Efficient aggregation and feature generation using distributed DataFrames
- Parallel training of RandomForest models across multiple executors
- Faster experimentation with multiple hyperparameter settings

This made it possible to train and evaluate models at a scale that would be significantly slower on a local machine. 

---
## Task 1 — Second model + dimensionality reduction (3 pts)

**TODO:** DR (PCA / SVD / Ray) + follow-up (supervised recommended). Fit on train only.


---
## Task 2 — Evaluate your model (3 pts)

**TODO:** Model 2 train vs test; explained variance; silhouette if clustering.


---
## Task 3 — Fitting analysis (3 pts)

**TODO:** Fitting graph; improvements; DR vs full features.


---
## Task 4 — README (1 pt)

**TODO:** `README.md` + links to `milestone_4.ipynb`, branch, scripts.


---
## Task 5 — Conclusion (3 pts)

**TODO:** Methods (M1+M2); Conclusion both models; second model + how to improve.


---
## Task 6 — Predictions analysis — test only (2 pts)

**TODO:** Test set — correct, FP, FN.


---
Stop Spark when finished.


In [ ]:
spark.stop()
